# 1. Introduction

Natural Disaster explores multilabel classification of English disaster messages. A message can describe several needs at once, such as food, water and shelter.

The walkthrough covers CSV preparation, SQLite storage, text preprocessing, grouped evaluation and an XGBoost model. Local scripts are in `src/`; setup instructions are in `README.md`.

Reference: [Sajal's Disaster Message Classifier walkthrough](https://github.com/sajal2692/disaster-message-classifier). The notebook contains material adapted from that learning project.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 2. Data

Data is available to us in two different files: `disaster_messages.csv` and `disaster_categories.csv`.

Let's load these two datasets and take a look at their first few rows.


In [ ]:
categories = pd.read_csv('data/disaster_categories.csv')
messages = pd.read_csv('data/disaster_messages.csv')

In [ ]:
messages.head()

We're only interested in the `message` column from the `disaster_messages` dataset. We'll use that column to train our message classifier, and ignore the other columns.

In [ ]:
categories.head()

In [ ]:
categories.iloc[0]['categories']

The `disaster_categories` dataset contains the category labels for our messages, but in a serialized fashion. We have also taken a closer look at the category for the first row. We'll need to convert the categories to a format better suited for our ML model.

# 3. Extract, Transform, Load!

Let's start by merging the two datasets so that the messages and categories are present in the same dataframe.

In [ ]:
df = messages.merge(categories, on='id')

In [ ]:
df.head()

### Cleaning and transforming the categories column 

Let's start by splitting the  categories column into separate columns for each category.

In [ ]:
categories = df['categories'].str.split(';',expand=True)

In [ ]:
categories.head()

Extracting the column names from a row of the categories dataframe.

In [ ]:
category_column_names = categories.iloc[0].apply(lambda x: x.split("-")[0])

In [ ]:
# Let's check which categories we have
print([name for name in category_column_names])

Using the extracted column names as the header of the categories dataframe:

In [ ]:
categories.columns = category_column_names

In [ ]:
categories.head()

Next, we'll fix the values of the above dataset so that they are binary. 1 indicates that a message belonging to a given category.

In [ ]:
for column in categories:
    # the last character of a value indicates its binary label
    categories[column] = categories[column].str[-1].astype(int)

The `related` column includes 0, 1 and 2. This learning workflow maps 2 to 1. This is a preprocessing assumption, not a verified interpretation of the annotation scheme.

In [ ]:
categories['related'].value_counts()

In [ ]:
categories['related'] = categories['related'].clip(0,1)

In [ ]:
categories['related'].value_counts()

Let's concatenate the categories dataframe back to our original dataframe.

In [ ]:
# drop the original categories column from `df`
df.drop('categories',axis=1,inplace=True)

# concatenate the original dataframe with the new `categories` dataframe
df = pd.concat([df,categories], axis=1)


In [ ]:
df.head()

### Removing duplicates and NaN rows, if any

In [ ]:
# checking for duplicate rows in df
df.duplicated(subset="message",keep=False).value_counts()

In [ ]:
# drop duplicates
df.drop_duplicates(keep='first', inplace=True)

In [ ]:
# drop NaN messages
df.dropna(subset=["message"], axis=0, inplace=True) # drop the row

In [ ]:
# drop the id and original columns as they are not useful for the learning problem
df.drop(["id", "original"], axis=1, inplace=True)

### Saving to a database

As part of this project's simulated scenario, we'll use sqlite and save the dataframe to a db.

In [ ]:
database_filename = "disaster_db"

engine = create_engine('sqlite:///'+ database_filename)
df.to_sql('messages', engine, if_exists="replace", index=False)

# 4. Machine Learning Pipeline

Let's load the data back from our database. The target `child_alone` has no positive examples in this dataset and is excluded from model training.

In [ ]:
import re

import nltk
# uncomment below if NLTK data needs to be downloaded
# nltk.download(['punkt', 'punkt_tab', 'wordnet', 'omw-1.4'])
# nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

from xgboost import XGBClassifier

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.multioutput import MultiOutputClassifier

In [ ]:
engine = create_engine('sqlite:///' + database_filename)
df = pd.read_sql_query('select * from messages', engine)
# the training data is a numpy array of all messages
X = df['message'].values
# the labels are all the different categories
Y = df.drop(columns=['message', 'genre', 'child_alone'])
category_names = Y.columns

In [ ]:
len(X)

In [ ]:
X[:5]

In [ ]:
Y.head()

### Splitting the dataset into train/test sets

Use a reproducible 80/20 group split so identical message texts cannot occur in both partitions. Similar wording and shared disaster events are not necessarily kept together.


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, Y, groups=X))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]
assert not (set(X_train) & set(X_test))

In [ ]:
print(len(X_train), len(X_test))

### Preprocessing the messages texts

For the classifier to perform well, we need to preprocess the disaster message texts to standardize and tokenize them. We can define a function to do this, which can be used later when defining our model pipeline:

In [ ]:
punctuation_regex = re.compile(r"[^\w\s]")
stop_words = set(stopwords.words('english'))
wordnet_lemmatizer = WordNetLemmatizer()
pos_tags_to_lemmatize = ["n", "v"]

In [ ]:
def tokenize(text):
    """
    Tokenizes a given text.
    Args:
        text: text string
    Returns:
        tokens: list of tokens
    """
    # lowercase string and remove punctuation
    text = punctuation_regex.sub(" ", text.lower()).strip()
    # tokenize text
    tokens = [token for token in word_tokenize(text)]
    # lemmatize text based on pos tags
    for pos_tag in pos_tags_to_lemmatize:
        tokens = [wordnet_lemmatizer.lemmatize(token, pos=pos_tag) for token in tokens]
    # remove stopwords
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

### Building the model pipeline

In [ ]:
def build_model():
    """Builds classification model """

    pipeline = Pipeline([
        ('vect', CountVectorizer(tokenizer=tokenize, token_pattern=None)),
        ('tfidf', TfidfTransformer()),
        ('clf', MultiOutputClassifier(XGBClassifier(learning_rate=0.1, n_jobs=1, random_state=42)))
    ])

    parameters = {
        "clf__estimator__max_depth": [8, 16],
        "clf__estimator__colsample_bytree":[0.5, 0.75]
    }

    cv = GridSearchCV(pipeline, cv=GroupKFold(n_splits=3), param_grid=parameters, n_jobs=1, scoring="f1_micro")
    return cv

### Fitting the model

In [ ]:
model = build_model()

In [ ]:
model.fit(X_train, y_train, groups=X_train)

### Evaluating the model

Report true labels first and predicted labels second. Micro-F1 combines all label decisions; macro-F1 gives equal weight to each category.


In [ ]:
y_preds = model.predict(X_test)
print(classification_report(y_test.values, y_preds, target_names=category_names, zero_division=0))

In [ ]:
# collect accuracy scores in a dict
category_name_2_accuracy_score = {}
for i in range(len(category_names)):
    category_name_2_accuracy_score[y_test.columns[i]] = accuracy_score(y_test.values[:,i],y_preds[:,i])
print(pd.Series(category_name_2_accuracy_score))

This is an imbalanced dataset, as most categories for a given message will be 0. The f1-score is a better metric of the model's performance.

### Predicting categories for new messages

In [ ]:
def predict(text):
    """Returns a list of predicted categories for the given text"""
    preds = model.predict([text])
    predicted_categories = [category for i, category in enumerate(y_test.columns) if preds[0][i] == 1]
    return predicted_categories

In [ ]:
predict("after the floods in our area we are trapped. we need food and shelter ")

# 5. Further Improvements


This notebook sticks to the basics in order to provide a good baseline model for this classification task. There is plenty of room for improvement here, including but not limited to:

1. Using word embeddings (GloVe, word2vec) or even sentence embeddings (Univsersal Sentgence Encoder) to transform message text, instead of using a CountVectorizer. This should allow the model to generalize well to similar/unseen words and improve the model accuracy too.
2. Some of the category column values like `related` and `child_alone` are highly skewed. We can look at adding more data for these categories.
3. A Neural Network can be trained for this task. Better yet, a pre-trained state-of-the-art transformer based network can be fine tuned using the data available to us.
4. Certain classification categories are "noisy" such as "related" or "child alone" (no positive examples). These can either be removed, or more data can be acquired that provides enough missing examples for such cases.

# 6. Summary

In this project, we built an ETL pipeline to load messy and unsuitable-for-training data, clean and transform it, and save it to a database. We also built a ML pipeline to tokenize message text, and train an XGBoost classifier to classify messages into different categories. 

We stuck to the basics for building this classifier, and there's plenty of room for improvement in the future using modern NLP architectures.